# EfficientNet-B0 Respiratory Sound Classification

Train a pretrained EfficientNet-B0 on 128x128 mel spectrograms and evaluate at patient level using macro F1.

In [24]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix, f1_score, classification_report
 )

import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cpu


## 1) Load Metadata and Validate Spectrogram Paths
Read processed_audio/metadata.csv, verify columns, and ensure spectrogram paths exist.

In [ ]:
from pathlib import Path
import pandas as pd

# Ajuste aqui se você estiver lendo direto do Drive
possible_roots = [
    Path("/content/processed_audio"),                # se você copiou para o disco local
    Path("/content/drive/MyDrive/processed_audio"),  # se estiver lendo direto do Drive
]

project_root = None
for root in possible_roots:
    if (root / "metadata.csv").exists():
        project_root = root.parent
        break

if project_root is None:
    raise FileNotFoundError(
        "Não encontrei processed_audio/metadata.csv nem em /content nem no Drive."
    )

metadata_path = project_root / "processed_audio" / "metadata.csv"

df = pd.read_csv(metadata_path)

required_cols = [
    "patient_id",
    "spectrogram_path",
    "diagnosis",
    "split",
    "crackles",
    "wheezes",
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df.copy()

# Remove nulos antes de converter para string
df = df.dropna(subset=["patient_id", "spectrogram_path", "diagnosis", "split"])

df["diagnosis"] = df["diagnosis"].astype(str).str.strip()
df["split"] = df["split"].astype(str).str.strip()
df["spectrogram_path"] = df["spectrogram_path"].astype(str).str.strip()

# Remove linhas vazias ou inválidas
df = df[(df["diagnosis"] != "") & (df["diagnosis"].str.lower() != "nan")]

# Resolve caminhos relativos a partir da pasta processed_audio
df["spec_path_abs"] = df["spectrogram_path"].map(lambda p: (project_root / "processed_audio" / p).resolve())

missing_paths = df[~df["spec_path_abs"].map(lambda p: p.exists())]
if not missing_paths.empty:
    raise FileNotFoundError(
        f"Missing spectrogram files. Examples: {missing_paths['spectrogram_path'].head(5).tolist()}"
    )

print("Metadata rows:", len(df))
print("Project root:", project_root)

In [25]:
cwd = Path.cwd().resolve()
project_root = cwd if (cwd / 'processed_audio').exists() else cwd.parent
metadata_path = project_root / 'processed_audio' / 'metadata.csv'

if not metadata_path.exists():
    raise FileNotFoundError(f'Metadata file not found: {metadata_path}')

df = pd.read_csv(metadata_path)
required_cols = [
    'patient_id', 'spectrogram_path', 'diagnosis', 'split', 'crackles', 'wheezes'
 ]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

df = df.copy()
df['diagnosis'] = df['diagnosis'].astype(str).str.strip()
df['split'] = df['split'].astype(str)
df['spectrogram_path'] = df['spectrogram_path'].astype(str)
df = df[df['diagnosis'] != '']

df['spec_path_abs'] = df['spectrogram_path'].map(lambda p: project_root / str(p))
missing_paths = df[~df['spec_path_abs'].map(lambda p: p.exists())]
if not missing_paths.empty:
    raise FileNotFoundError(
        f"Missing spectrogram files. Examples: {missing_paths['spectrogram_path'].head(5).tolist()}"
    )

print('Metadata rows:', len(df))

Metadata rows: 6898


## 2) Build Patient-Level Splits from Metadata
Use the split column and verify no patient overlap across splits.

In [26]:
splits = ['train', 'validation', 'test']
df_splits = {s: df[df['split'] == s].copy() for s in splits}
for s in splits:
    if df_splits[s].empty:
        raise ValueError(f'No samples found for split: {s}')

split_counts = df[['patient_id', 'split']].drop_duplicates().groupby('patient_id')['split'].nunique()
leaky_patients = split_counts[split_counts > 1]
print('Patients appearing in multiple splits:', len(leaky_patients))
if len(leaky_patients) > 0:
    display(leaky_patients.head(10))

print('Train/Val/Test sizes:', {k: len(v) for k, v in df_splits.items()})

Patients appearing in multiple splits: 0
Train/Val/Test sizes: {'train': 4503, 'validation': 1288, 'test': 1107}


## 3) Label Encoding and Class Mapping
Fit a label encoder on diagnosis labels and store mapping.

In [27]:
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['diagnosis'])
label_mapping = {label: int(idx) for idx, label in enumerate(label_encoder.classes_)}
inv_label_mapping = {int(idx): label for label, idx in label_mapping.items()}
df_splits = {s: df[df['split'] == s].copy() for s in splits}

display(pd.DataFrame({'label': list(label_mapping.keys()), 'index': list(label_mapping.values())}))

,label,index
0,Bronchiectasis,0
1,Bronchiolitis,1
2,COPD,2
3,Healthy,3
4,Other,4
5,Pneumonia,5
6,URTI,6


## 4) Dataset: On-Demand .npy Loading + Preprocessing
Load spectrograms on demand, normalize, and adapt to 3-channel 128x128 input.

In [28]:
def normalize_spectrogram(arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32)
    arr_min = arr.min()
    arr_max = arr.max()
    return (arr - arr_min) / (arr_max - arr_min + 1e-6)

def to_3ch_tensor(arr: np.ndarray) -> torch.Tensor:
    if arr.shape == (128, 128):
        arr = np.expand_dims(arr, axis=0)
    elif arr.shape == (128, 128, 1):
        arr = np.transpose(arr, (2, 0, 1))
    if arr.shape != (1, 128, 128):
        raise ValueError(f'Unexpected spectrogram shape: {arr.shape}')
    arr = np.repeat(arr, 3, axis=0)
    return torch.from_numpy(arr)

class SpectrogramDataset(Dataset):
    def __init__(self, df_split: pd.DataFrame, *, training: bool, augment_fn=None):
        self.df = df_split.reset_index(drop=True)
        self.training = training
        self.augment_fn = augment_fn

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        arr = np.load(row['spec_path_abs'], allow_pickle=False)
        arr = normalize_spectrogram(arr)
        if self.training and self.augment_fn is not None:
            arr = self.augment_fn(arr)
        tensor = to_3ch_tensor(arr)
        label = int(row['label'])
        pid = row['patient_id']
        return tensor, label, pid

## 5) Augmentation: SpecAugment (Train Only)
Apply light random time and frequency masking for training samples.

In [29]:
def spec_augment(arr: np.ndarray, time_mask_max=12, freq_mask_max=12) -> np.ndarray:
    aug = arr.copy()
    freq_len, time_len = aug.shape[-2], aug.shape[-1]
    if time_len > 1:
        t = np.random.randint(0, min(time_mask_max, time_len))
        t0 = np.random.randint(0, max(1, time_len - t))
        aug[..., t0:t0 + t] = 0.0
    if freq_len > 1:
        f = np.random.randint(0, min(freq_mask_max, freq_len))
        f0 = np.random.randint(0, max(1, freq_len - f))
        aug[..., f0:f0 + f, :] = 0.0
    return aug

## 6) Dataloaders and Collation
Build DataLoaders with shuffling for training only.

In [30]:
batch_size = 32 if torch.cuda.is_available() else 16
num_workers = 2 if torch.cuda.is_available() else 0

train_ds = SpectrogramDataset(df_splits['train'], training=True, augment_fn=spec_augment)
val_ds = SpectrogramDataset(df_splits['validation'], training=False, augment_fn=None)
test_ds = SpectrogramDataset(df_splits['test'], training=False, augment_fn=None)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=torch.cuda.is_available())

print('Batches (train/val/test):', len(train_loader), len(val_loader), len(test_loader))

Batches (train/val/test): 282 81 70


## 7) Model: EfficientNet-B0 Backbone + Classification Head
Load pretrained EfficientNet-B0 and replace the classifier head.

In [31]:
import ssl
import certifi

ssl._create_default_https_context = ssl.create_default_context
ssl._create_default_https_context = ssl.create_default_context(cafile=certifi.where())

In [32]:
num_classes = len(label_encoder.classes_)
local_weights_path = project_root / 'processed_audio' / 'models' / 'efficientnet_b0_rwightman-7f5810bc.pth'

if local_weights_path.exists():
    model = torchvision.models.efficientnet_b0(weights=None)
    state = torch.load(local_weights_path, map_location='cpu')
    if isinstance(state, dict) and 'state_dict' in state:
        state = state['state_dict']
    cleaned_state = {
        k.replace('module.', '').replace('model.', ''): v for k, v in state.items()
    }
    missing, unexpected = model.load_state_dict(cleaned_state, strict=False)
    print('Loaded local weights:', local_weights_path)
    print('Missing keys:', len(missing), 'Unexpected keys:', len(unexpected))
else:
    weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
    model = torchvision.models.efficientnet_b0(weights=weights)

in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, num_classes)
 )
model = model.to(device)
print(model.classifier)

Loaded local weights: /Users/guilhermekaidei/Desktop/Insper/AI_Medicine/Project/processed_audio/models/efficientnet_b0_rwightman-7f5810bc.pth
Missing keys: 0 Unexpected keys: 0
Sequential(
  (0): Dropout(p=0.4, inplace=False)
  (1): Linear(in_features=1280, out_features=7, bias=True)
)


## 8) Loss, Optimizer, and Metrics Setup
Define Focal Loss, AdamW, and helper metric functions.

In [33]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        logp = nn.functional.log_softmax(logits, dim=1)
        p = torch.exp(logp)
        pt = p.gather(1, targets.unsqueeze(1)).squeeze(1)
        loss = -(1 - pt) ** self.gamma * logp.gather(1, targets.unsqueeze(1)).squeeze(1)
        if self.reduction == 'mean':
            return loss.mean()
        if self.reduction == 'sum':
            return loss.sum()
        return loss

criterion = FocalLoss(gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

def compute_patient_metrics(results_df, label_order):
    y_true = results_df['true_label'].values
    y_pred = results_df['pred_label'].values
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=label_order)
    return acc, precision, recall, f1, cm

## 9) Stage 1 Training: Frozen Backbone
Train the classification head with early stopping on patient-level macro F1.

In [ ]:
def predict_cycle_probs(model, loader):
    model.eval()
    all_probs = []
    all_labels = []
    all_pids = []
    with torch.no_grad():
        for xb, yb, pids in loader:
            xb = xb.to(device)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(yb.numpy())
            all_pids.extend(pids)
    probs = np.concatenate(all_probs, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    cycle_df = pd.DataFrame(probs, columns=label_encoder.classes_)
    cycle_df['patient_id'] = all_pids
    cycle_df['true_label'] = label_encoder.inverse_transform(labels)
    return cycle_df

def aggregate_patient_results(cycle_df):
    patient_probs = cycle_df.groupby('patient_id')[label_encoder.classes_].mean()
    patient_true = cycle_df.groupby('patient_id')['true_label'].agg(lambda x: x.iloc[0])
    patient_pred = patient_probs.idxmax(axis=1)
    patient_results = pd.DataFrame({
        'patient_id': patient_probs.index,
        'true_label': patient_true.values,
        'pred_label': patient_pred.values,
        'max_confidence': patient_probs.max(axis=1).values
    })
    return patient_results

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for xb, yb, _ in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

def evaluate_loss(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for xb, yb, _ in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

def run_training_stage(model, train_loader, val_loader, optimizer, criterion, max_epochs, patience, stage_name, best_ckpt_path):
    history = {'train_loss': [], 'val_loss': [], 'val_f1': []}
    best_f1 = -np.inf
    best_state = None
    patience_left = patience

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss = evaluate_loss(model, val_loader, criterion)
        val_cycle_df = predict_cycle_probs(model, val_loader)
        val_patient_results = aggregate_patient_results(val_cycle_df)
        _, _, _, val_f1, _ = compute_patient_metrics(val_patient_results, label_encoder.classes_)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)

        print(f'{stage_name} Epoch {epoch:02d} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_f1={val_f1:.4f}')

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save({'model_state': model.state_dict(), 'best_f1': best_f1}, best_ckpt_path)
            patience_left = patience
        else:
            patience_left -= 1
            if patience_left <= 0:
                print('Early stopping triggered.')
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return history, best_f1

for param in model.features.parameters():
    param.requires_grad = False

stage1_ckpt = project_root / 'processed_audio' / 'models' / 'efficientnet_b0_stage1.pt'
stage1_ckpt.parent.mkdir(parents=True, exist_ok=True)

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)
stage1_history, stage1_best_f1 = run_training_stage(
    model, train_loader, val_loader, optimizer, criterion,
    max_epochs=10, patience=3, stage_name='Stage1', best_ckpt_path=stage1_ckpt
 )

Stage1 Epoch 01 | train_loss=0.4872 val_loss=0.4509 val_f1=0.1630
Stage1 Epoch 02 | train_loss=0.4219 val_loss=0.4152 val_f1=0.1815
Stage1 Epoch 03 | train_loss=0.4105 val_loss=0.4007 val_f1=0.1784
Stage1 Epoch 04 | train_loss=0.3950 val_loss=0.4028 val_f1=0.1599
Stage1 Epoch 05 | train_loss=0.3903 val_loss=0.3984 val_f1=0.1569
Early stopping triggered.


## 10) Stage 2 Fine-Tuning: Unfreeze Backbone
Unfreeze the backbone and continue training with a lower learning rate.

In [35]:
for param in model.features.parameters():
    param.requires_grad = True

stage2_ckpt = project_root / 'processed_audio' / 'models' / 'efficientnet_b0_stage2.pt'
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

stage2_history, stage2_best_f1 = run_training_stage(
    model, train_loader, val_loader, optimizer, criterion,
    max_epochs=15, patience=4, stage_name='Stage2', best_ckpt_path=stage2_ckpt
 )

best_stage = 'stage2' if stage2_best_f1 >= stage1_best_f1 else 'stage1'
best_ckpt = stage2_ckpt if best_stage == 'stage2' else stage1_ckpt
ckpt = torch.load(best_ckpt, map_location=device)
model.load_state_dict(ckpt['model_state'])
print('Best checkpoint:', best_ckpt, 'best_f1=', ckpt['best_f1'])

Stage2 Epoch 01 | train_loss=0.3335 val_loss=0.3260 val_f1=0.1715
Stage2 Epoch 02 | train_loss=0.2057 val_loss=0.3292 val_f1=0.2387
Stage2 Epoch 03 | train_loss=0.1351 val_loss=0.3618 val_f1=0.3242


KeyboardInterrupt: 

## 11) Cycle-Level Inference and Patient-Level Aggregation
Compute per-cycle probabilities and aggregate by patient.

In [ ]:
val_cycle_df = predict_cycle_probs(model, val_loader)
test_cycle_df = predict_cycle_probs(model, test_loader)

val_patient_results = aggregate_patient_results(val_cycle_df)
test_patient_results = aggregate_patient_results(test_cycle_df)

val_patient_results.head()

## 12) Evaluation: Macro F1, Precision, Recall, Accuracy, Confusion Matrix
Report patient-level metrics and plot confusion matrices for validation and test.

In [ ]:
val_acc, val_precision, val_recall, val_f1, val_cm = compute_patient_metrics(
    val_patient_results, label_encoder.classes_
 )
test_acc, test_precision, test_recall, test_f1, test_cm = compute_patient_metrics(
    test_patient_results, label_encoder.classes_
 )

print('Validation patient-level accuracy:', val_acc)
print('Validation patient-level precision (macro):', val_precision)
print('Validation patient-level recall (macro):', val_recall)
print('Validation patient-level F1 (macro):', val_f1)
print('Test patient-level accuracy:', test_acc)
print('Test patient-level precision (macro):', test_precision)
print('Test patient-level recall (macro):', test_recall)
print('Test patient-level F1 (macro):', test_f1)

print('\nValidation classification report:')
print(classification_report(
    val_patient_results['true_label'],
    val_patient_results['pred_label'],
    zero_division=0
 ))

print('Test classification report:')
print(classification_report(
    test_patient_results['true_label'],
    test_patient_results['pred_label'],
    zero_division=0
 ))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(
    val_cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_, ax=axes[0]
 )
axes[0].set_title('Validation confusion matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
sns.heatmap(
    test_cm, annot=True, fmt='d', cmap='Greens',
    xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_, ax=axes[1]
 )
axes[1].set_title('Test confusion matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
plt.tight_layout()
plt.show()

## 13) Learning Curves and Diagnostics
Plot training and validation loss and macro F1 for both stages.

In [ ]:
epochs_stage1 = list(range(1, len(stage1_history['train_loss']) + 1))
epochs_stage2 = list(range(1, len(stage2_history['train_loss']) + 1))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(epochs_stage1, stage1_history['train_loss'], label='train_s1')
axes[0].plot(epochs_stage1, stage1_history['val_loss'], label='val_s1')
axes[0].plot(epochs_stage2, stage2_history['train_loss'], label='train_s2')
axes[0].plot(epochs_stage2, stage2_history['val_loss'], label='val_s2')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs_stage1, stage1_history['val_f1'], label='val_f1_s1')
axes[1].plot(epochs_stage2, stage2_history['val_f1'], label='val_f1_s2')
axes[1].set_title('Validation Macro F1')
axes[1].set_xlabel('Epoch')
axes[1].legend()

axes[2].bar(['stage1_best', 'stage2_best'], [stage1_best_f1, stage2_best_f1])
axes[2].set_title('Best Val F1')
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 14) Checkpointing and Artifact Saving
Save the best checkpoint, label mapping, and run metadata.

In [ ]:
artifacts_dir = project_root / 'processed_audio' / 'artifacts'
artifacts_dir.mkdir(parents=True, exist_ok=True)

label_map_path = artifacts_dir / 'label_mapping.json'
with open(label_map_path, 'w', encoding='utf-8') as f:
    json.dump(label_mapping, f, indent=2)

run_info = {
    'seed': SEED,
    'batch_size': batch_size,
    'stage1_best_f1': float(stage1_best_f1),
    'stage2_best_f1': float(stage2_best_f1),
    'best_checkpoint': str(best_ckpt)
}
run_info_path = artifacts_dir / 'run_info.json'
with open(run_info_path, 'w', encoding='utf-8') as f:
    json.dump(run_info, f, indent=2)

print('Saved label mapping to:', label_map_path)
print('Saved run info to:', run_info_path)